# IMPORTANT: Execution Instructions

- **Environment**: This project was developed for the Kaggle environment. It relies on Kaggle's specific directory structure (`/kaggle/input` and `/kaggle/working`).

- **Modular Execution**: Due to GPU VRAM limits, the evaluation sections are modular. It is not possible to load all 4 models simultaneously. Please restart the kernel or use the VRAM clearing function provided at the end of the notebook before switching between models (for example, when switching from Llama to Qwen).

- **External Data & Checkpoints**: All required files (CSV data, model checkpoints, and QLoRA adapters) are hosted in the following Kaggle Dataset: [https://kaggle.com/datasets/ca50c21af42a96eb8fffec28470b023a0c551ab9a874cdb9db295c16baa43945].

- **Reproducibility**: To reproduce the results, add the dataset above to your Kaggle session. The code is already configured to access these files via the `/kaggle/input` path.

- **Model selection**: It is possible to choose the name of the evaluated model in the main pipeline using the 'model_name' variable. Don't forget to first load the model.

# 1. Environment Setup and Library Imports
Importing necessary libraries and defining global constants like `RADGRAPH_METRIC_LABELS`.

In [ ]:
!pip3 install --upgrade pip
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q peft

In [ ]:
import json
import pandas as pd
import numpy as np
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, TrainingArguments, Trainer
from typing import List, Dict, Any, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
import json
import re
import ast
warnings.filterwarnings('ignore')

In [ ]:
# Labels used for multi-label classification problem
RADGRAPH_METRIC_LABELS = [
    "normal","clear","sharp","sharply","unremarkable","intact","stable","free","effusion",
    "opacity","pneumothorax","edema","atelectasis","tube","consolidation","process",
    "abnormality","enlarge","tip","low","pneumonia","line","congestion","catheter",
    "cardiomegaly","fracture","air","tortuous","lead","disease","calcification",
    "prominence","device","engorgement","picc","clip","elevation","expand","nodule",
    "wire","fluid","degenerative","pacemaker","thicken","marking","scar","hyperinflate",
    "blunt","loss","widen","collapse","density","emphysema","aerate","mass","crowd",
    "infiltrate","obscure","deformity","hernia","drainage","distention","shift","stent",
    "pressure","lesion","finding","borderline","hardware","dilation","chf",
    "redistribution","aspiration","tail_abnorm_obs","excluded_obs"
]

# --- AUXILIARY FUNCTIONS ---
def _parse_class_list_any(obj):
    """
    Parse class_list into (labels, class_names)
    - labels: list[int] of 0/1
    - class_names: list[str] or None when unnamed list input

    Accepts: str (JSON), dict{name: val}, list[val]
    """
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except Exception:
            return [], None
    if isinstance(obj, dict):
        keys = list(obj.keys())
        vals = [obj[k] for k in keys]
        labels = [1 if int(round(float(v))) == 1 else 0 for v in vals]
        return labels, keys
    if isinstance(obj, list):
        try:
            labels = [1 if int(round(float(v))) == 1 else 0 for v in obj]
        except Exception:
            labels = [1 if v == 1 else 0 for v in obj]
        return labels, None
    return [], None

def extract_raw_caption(row):
    """Extracts the text from the JSON report"""
    try:
        umls_data = json.loads(row['umls_json_info'])
        if 'caption' in umls_data:
            cap = umls_data['caption']
            if isinstance(cap, list):
                return ' '.join(map(str, cap))
            return str(cap)
        return ""
    except Exception:
        return ""

def simple_clean(text):
    """Basic text cleaning"""
    return text.lower().replace('.', ' ').replace(',', ' ').strip()

def load_data(csv_path="/kaggle/input/mimic-cxr-2/sampled_1000_data.csv"):
    """Loads and processes data"""
    print("Loading data...")
    df = pd.read_csv(csv_path, engine='python', on_bad_lines='skip', encoding='utf-8')

    # Extract captions
    df['raw_captions'] = df.apply(extract_raw_caption, axis=1)

    # Process labels
    parsed_labels = []
    for _, row in df.iterrows():
        labels, _ = _parse_class_list_any(row.get('class_list', []))
        parsed_labels.append(labels)

    # Create labels matrix (75 classes)
    y_binary_list = []
    for labels in parsed_labels:
        vec = [0] * 75
        for i, val in enumerate(labels[:75]):
            vec[i] = val
        y_binary_list.append(vec)

    y_binary = np.array(y_binary_list, dtype=int)

    print(f"Loaded {len(df)} registers")
    print(f"   Labels' shape: {y_binary.shape}\n")

    return df['raw_captions'].tolist(), y_binary

def prepare_training_data(reports, y_labels):
    """Prepares data in the format (prompt, response) for fine-tuning."""
    training_examples = []
    
    for report, label_vec in zip(reports, y_labels):
        #  Get positive labels
        positive_labels = [RADGRAPH_METRIC_LABELS[i] for i in range(len(label_vec)) if label_vec[i] == 1]
        
        # Expected response
        response = str(positive_labels)

        # Create prompt (same as zero-shot)
        prompt = create_zero_shot_prompt(report)

        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response}
        ]
        
        # The tokenizer applies the correct format
        full_text = tokenizer.apply_chat_template(messages, tokenize=False)
        
        training_examples.append({"text": full_text})
    
    return training_examples

def tokenize_function(examples):
    """Tokenize examples for the model"""
    tokenized = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

# 2. Subtask 1: Classical Baseline Model
This section implements the classical baseline. We process the text, vectorize it using TF-IDF, train a LinearSVC with 200 instances from the labeled dataset and evaluate with 800 instances.

In [ ]:
def build_baseline_data(csv_path: str = "./data/sampled_1000_data.csv"):
    print("1. Loading data...")
    try:
        df = pd.read_csv(csv_path, engine='python', on_bad_lines='skip', encoding='utf-8')
    except Exception as e:
        raise RuntimeError(f"Error reading CSV: {e}")

    print(f"   -> Loaded {len(df)} records.")

    print("2. Processing text...")
    df['raw_captions'] = df.apply(extract_raw_caption, axis=1)
    df['processed_captions'] = df['raw_captions'].apply(simple_clean)

    print("3. Processing labels...")

    # 3.1 Decoding each line
    parsed_labels = []
    for _, row in df.iterrows():
        labels, _ = _parse_class_list_any(row.get('class_list', []))
        parsed_labels.append(labels)

    # 3.2 Defining fixed size (C)
    # For our project, we will use the 75 classes from RadGraph Metric Labels
    C = 75

    # 3.3 Build the matrix Y filled with zeros
    y_binary_list = []
    for labels in parsed_labels:
        vec = [0] * C
        # Fill in with the values that came from the JSON (truncating if there are more than 75)
        for i, val in enumerate(labels[:C]):
            vec[i] = val
        y_binary_list.append(vec)

    y_binary = np.array(y_binary_list, dtype=int)

    # 4. Vectorization (X)
    print("4. Vectorizing text (TF-IDF)...")
    vectorizer = TfidfVectorizer(stop_words='english', max_features=1000, ngram_range=(1, 2))
    X_tfidf = vectorizer.fit_transform(df['processed_captions'].fillna(''))

    return X_tfidf, y_binary, RADGRAPH_METRIC_LABELS

# --- EXECUTION ---

X_features, y_labels, class_names = build_baseline_data(csv_path="/kaggle/input/mimic-cxr-2/sampled_1000_data.csv")

print("\n--- DATA SUMMARY ---")
print(f"X (Features) Shape: {X_features.shape}")
print(f"Y (Labels) Shape:   {y_labels.shape}")

X_train, X_test, y_train, y_test = train_test_split(X_features, y_labels, test_size=0.2, random_state=42)

print("\n--- BASELINE SETUP ---")
print("Training OneVsRestClassifier (This may take a few seconds)...")

# Trains the model
clf_logres = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
clf_svm = OneVsRestClassifier(LinearSVC(
    loss='squared_hinge',
    dual=False,
    max_iter=3000,
    random_state=42
))
clf_svm.fit(X_train, y_train)

# Prediction and Metrics
print("Evaluating...")
y_pred = clf_svm.predict(X_test)
macro_f1_svc = f1_score(y_test, y_pred, average='macro')
micro_f1_svc = f1_score(y_test, y_pred, average='micro')

print(f"\nFINAL RESULT:")
print(f"Macro-F1 Score: {macro_f1_svc:.4f} , Micro-F1 Score: {micro_f1_svc:.4f}")

# 3. Subtask 3: Base decoder models (0-shot and 5-shot) and QLoRA Fine-tuned models
We define functions and templates for evaluating decoder models under 0-shot and 5-shot configurations and also for fine-tuning models using QLoRA parameters.

In [ ]:
# Copying checkpoints from input to working folder
# (necessary if there are still instances to be infered and added to the checkpoint)
!cp /kaggle/input/mimic-cxr-2/model_checkpoints/checkpoint_0-Shot-llama.pkl /kaggle/working/checkpoint_0-Shot-qwen-3b.pkl
!cp /kaggle/input/mimic-cxr-2/model_checkpoints/checkpoint_5-Shot-llama.pkl /kaggle/working/checkpoint_5-Shot-qwen-3b.pkl
!cp /kaggle/input/mimic-cxr-2/model_checkpoints/checkpoint_0-Shot-qwen-3b.pkl /kaggle/working/checkpoint_0-Shot-qwen-3b.pkl
!cp /kaggle/input/mimic-cxr-2/model_checkpoints/checkpoint_5-Shot-qwen-3b.pkl /kaggle/working/checkpoint_5-Shot-qwen-3b.pkl
!cp /kaggle/input/mimic-cxr-2/model_checkpoints/checkpoint_0-Shot-med-qwen.pkl /kaggle/working/checkpoint_0-Shot-med-qwen.pkl
!cp /kaggle/input/mimic-cxr-2/model_checkpoints/checkpoint_5-Shot-med-qwen.pkl /kaggle/working/checkpoint_5-Shot-med-qwen.pkl
!cp /kaggle/input/mimic-cxr-2/model_checkpoints/checkpoint_0-Shot-bio-llama.pkl /kaggle/working/checkpoint_0-Shot-bio-llama.pkl
!cp /kaggle/input/mimic-cxr-2/model_checkpoints/checkpoint_5-Shot-bio-llama.pkl /kaggle/working/checkpoint_5-Shot-bio-llama.pkl

In [ ]:
# ============================================================================
# PROMPT TEMPLATES
# ============================================================================

def create_zero_shot_prompt(report_text):
    """Template for zero-shot prompting - returns only positive labels."""
    labels_str = ", ".join(RADGRAPH_METRIC_LABELS)

    prompt = f"""Analyze this radiology report, extract medical findings from it and map them EXACTLY to the allowed label list.
    Return only a Python list, with NO EXPLANATION.
    
    Allowed labels (select only from this list): {labels_str}

    INSTRUCTIONS:
    1. Only return findings that are PRESENT. Do NOT return absent/negated findings (e.g., do not output "no pneumothorax").
    2. Output the EXACT label name from the list above. DO NOT add adjectives like "mild" or "small". 
       - Incorrect: "mild degenerative changes"
       - Correct: "degenerative"
       - Incorrect: "tortuosity"
       - Correct: "tortuous"
    3. Return a Python list of strings.
    4. Output format: ["label1", "label2"] or []
    
    Report: {report_text}
    
    Answer:"""

    return prompt

def create_five_shot_prompt(report_text, examples):
    """Template for 5-shot prompting - returns only positive labels."""
    labels_str = ", ".join(RADGRAPH_METRIC_LABELS)

    # Build examples (only positive labels)
    examples_text = ""
    for i, (ex_report, ex_labels) in enumerate(examples, 1):
        # Pegar apenas labels com valor 1
        positive_labels = [RADGRAPH_METRIC_LABELS[j] for j in range(len(ex_labels)) if ex_labels[j] == 1]
        examples_text += f"\nExample {i}:\nReport: {ex_report}\nPresent: {positive_labels}\n"

    prompt = f"""Analyze this radiology report, extract medical findings from it and map them EXACTLY to the allowed label list.
    Return only a Python list, with NO EXPLANATION.
    
    Allowed labels (select only from this list): {labels_str}

    INSTRUCTIONS:
    1. Only return findings that are PRESENT. Do NOT return absent/negated findings (e.g., do not output "no pneumothorax").
    2. Output the EXACT label name from the list above. DO NOT add adjectives like "mild" or "small". 
       - Incorrect: "mild degenerative changes"
       - Correct: "degenerative"
       - Incorrect: "tortuosity"
       - Correct: "tortuous"
    3. Return a Python list of strings.
    4. Output format: ["label1", "label2"] or []

    CONSIDER THE FOLLOWING EXAMPLES:
    {examples_text}
    
    Report: {report_text}
    
    Answer:"""

    return prompt

In [ ]:
# ============================================================================
# INFERENCE AND PARSING
# ============================================================================

def parse_model_output(response_text):
    """
    Robust version that reads both JSON and Python lists (single quotes) and handles truncated lists.
    """
    # 1. Clearing thought blocks
    text_to_parse = response_text
    if '<think>' in text_to_parse:
        text_to_parse = text_to_parse.split('</think>')[-1].strip()

    result = [0] * 75
    parsed_items = []

    # 2. Extracting the content between the brackets
    start = text_to_parse.find('[')
    
    if start != -1:
        end = text_to_parse.rfind(']')
        if end > start:
            # Full list
            list_str = text_to_parse[start:end+1]
        else:
            # List truncated (max_tokens reached): retrieve until the end
            list_str = text_to_parse[start:]
    else:
        # If the brackets are not found, try processing the entire text
        list_str = text_to_parse

    # 3. Decoding Attempts
    success = False

    # Attempt A: default JSON (double quotes)
    if not success:
        try:
            parsed_items = json.loads(list_str)
            success = True
        except:
            pass
            
    # Attempt B: literal Python (single quotes)
    if not success:
        try:
            parsed_items = ast.literal_eval(list_str)
            success = True
        except:
            pass
            
    # Attempt C: brute force (for truncated/poorly formatted lists)
    if not success:
        try:
            # Remove structural punctuation and separate with commas
            clean = list_str.replace('[','').replace(']','').replace("'", "").replace('"', "").replace('\n', ' ')
            parsed_items = [x.strip() for x in clean.split(',') if x.strip()]
            success = True
        except:
            parsed_items = []

    if not isinstance(parsed_items, list):
        return result

    # 4. Mapping for the 75 classes (flexible matching)
    for item in parsed_items:
        label_pred = str(item).strip().lower()

        # Ignore explicit negations (e.g. "no disease") as only positive labels are considered
        if re.search(r'\b(no|not|absence|negative)\b', label_pred):
            continue

        for i, ref_label in enumerate(RADGRAPH_METRIC_LABELS):
            target = ref_label.lower()
            # Regex without \b at the end to capture variations (e.g., enlarge -> enlargement)
            if re.search(r'\b' + re.escape(target), label_pred):
                result[i] = 1

    return result

# Inference function
def run_inference_with_checkpoint(reports, prompts, mode, model_name):
    import pickle
    import os

    # Checkpoints are used to save inference progress and prevent loss of outputs if the execution is interrupted for any reason
    checkpoint_file = f'/kaggle/working/checkpoint_{mode}-{model_name}.pkl'
    
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'rb') as f:
            predictions, last_idx = pickle.load(f)
        print(f"Resuming from index {last_idx}")
    else:
        predictions = []
        last_idx = 0
    
    print(f"Executing inference...")
    for i in tqdm(range(last_idx, len(reports))):
        messages = [{"role": "user", "content": prompts[i]}]
        
        try:
            response = pipe(messages, max_new_tokens=256, do_sample=False, repetition_penalty=1.1)
            full_text = response[0]['generated_text'][-1]['content']
            parsed = parse_model_output(full_text)
            print(full_text)
            print(parsed)
            predictions.append(parsed)
            
            # Save every 10 inferences
            if (i + 1) % 10 == 0:
                with open(checkpoint_file, 'wb') as f:
                    pickle.dump((predictions, i + 1), f)
            
        except Exception as e:
            print(f"⚠️ Erro: {e}")
            predictions.append([0] * 75)
    

    print(f"Inference completed!")
    
    return np.array(predictions)

In [ ]:
# ============================================================================
# EVALUATION
# ============================================================================

def evaluate_predictions(y_true, y_pred, mode):
    """Calculates Macro-F1 and Micro-F1 over LLM predictions"""
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)

    print(f"\n{'='*60}")
    print(f"RESULTS - {mode}")
    print(f"{'='*60}")
    print(f"Macro-F1 Score: {macro_f1:.4f}")
    print(f"Micro-F1 Score: {micro_f1:.4f}")
    print(f"{'='*60}\n")

    return {"mode": mode, "macro_f1": macro_f1, "micro_f1": micro_f1}

In [ ]:
# ============================================================================
# QLORA FINETUNING AND EVALUATION
# ============================================================================

def train_qlora():
    """Trains model with QLoRA"""
    print("\n" + "="*60)
    print("STARTING QLORA FINE-TUNING")
    print("="*60 + "\n")
    
    # 1. Load data
    reports, y_labels = load_data()
    reports_train, reports_test, y_train, y_test = train_test_split(
        reports, y_labels, test_size=0.2, random_state=42
    )
    
    # 2. Prepare model for k-bit training
    model_train = prepare_model_for_kbit_training(model)
    
    # 3. Configure LoRA
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model_train = get_peft_model(model_train, lora_config)
    
    print(f"Trainable parameters: {sum(p.numel() for p in model_train.parameters() if p.requires_grad):,}")
    
    # 4. Prepare dataset
    train_examples = prepare_training_data(reports_train, y_train)
    train_dataset = Dataset.from_list(train_examples)
    train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    
    # 5. Set up training
    training_args = TrainingArguments(
        output_dir="/kaggle/working/qlora_checkpoints",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        dataloader_num_workers=2,
        learning_rate=2e-4,
        fp16=True,
        bf16=False,
        logging_steps=5,
        logging_first_step=True,
        save_steps=100,
        save_total_limit=2,
        remove_unused_columns=False,
        report_to="none",
    )
    
    # 6. Train
    trainer = Trainer(
        model=model_train,
        args=training_args,
        train_dataset=train_dataset,
    )
    
    print("Starting training...\n")
    trainer.train()
    
    # 7. Save model
    model_train.save_pretrained("/kaggle/working/qlora_model")
    tokenizer.save_pretrained("/kaggle/working/qlora_model")
    print("Model saved!\n")
    
    # 8. Evaluate
    print("Evaluating fine-tuned model...\n")
    
    pipe_finetuned = pipeline("text-generation", model=model_train, tokenizer=tokenizer)
    
    predictions = []
    test_prompts = [create_zero_shot_prompt(r) for r in reports_test]

    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    
    terminators = [tokenizer.eos_token_id]
    if eot is not None:
        terminators.append(eot)
    if im_end is not None:
        terminators.append(im_end)
    
    for i, prompt in enumerate(tqdm(test_prompts)):
        formatted_prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True
        )
        try:
            response = pipe_finetuned(formatted_prompt, max_new_tokens=200, do_sample=False, eos_token_id=terminators, pad_token_id=tokenizer.eos_token_id)
            full_text = response[0]['generated_text']
            
            if formatted_prompt in full_text:
                full_text = full_text.replace(formatted_prompt, "").strip()
            
            predictions.append(parse_model_output(full_text))
        except Exception as e:
            print(f"Error {i}: {e}")
            predictions.append([0] * 75)
    
    predictions = np.array(predictions)
    
    # 9. Calculate metrics
    results_qlora = evaluate_predictions(y_test, predictions, "QLoRA Fine-tuned")
    
    return results_qlora

def evaluate_qlora_trained():
    """Evaluate the QLoRA model that has already been trained (no need to retrain it)."""
    print("\n" + "="*60)
    print("LOADING ALREADY TRAINED QLORA MODEL")
    print("="*60 + "\n")
    
    # 1. Load data
    reports, y_labels = load_data()
    reports_train, reports_test, y_train, y_test = train_test_split(
        reports, y_labels, test_size=0.2, random_state=42
    )
    
    # 2. Load trained model
    from peft import PeftModel

    # *** this is for llama final fine-tuned model, remember to change the directory for a different model! ***
    print("Loading model from /kaggle/input/mimic-cxr-2/qlora_model...")

    # *** in case you load model from checkpoints ***
    # print("Loading model from /kaggle/working/qlora_checkpoints/checkpoint-300...")
    
    # Load trained LoRa adapters
    model_trained = PeftModel.from_pretrained(model, "/kaggle/input/mimic-cxr-2/qlora_model") # using llama fine-tuned model

    # model_trained = PeftModel.from_pretrained(model, "/kaggle/working/qlora_checkpoints/checkpoint-300") # using model from checkpoints
    
    print("Model loaded successfully!\n")
    
    # 3. Evaluate
    print("Evaluating fine-tuned model...\n")
    
    pipe_finetuned = pipeline("text-generation", model=model_trained, tokenizer=tokenizer)
    
    predictions = []
    test_prompts = [create_zero_shot_prompt(r) for r in reports_test]
    
    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    
    terminators = [tokenizer.eos_token_id]
    if eot is not None:
        terminators.append(eot)
    if im_end is not None:
        terminators.append(im_end)
    
    for i, prompt in enumerate(tqdm(test_prompts)):
        formatted_prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True
        )
        
        try:
            response = pipe_finetuned(
                formatted_prompt,
                max_new_tokens=200,
                do_sample=False,
                eos_token_id=terminators,
                pad_token_id=tokenizer.eos_token_id
            )
            
            full_text = response[0]['generated_text']
            
            if formatted_prompt in full_text:
                full_text = full_text.replace(formatted_prompt, "").strip()
            
            predictions.append(parse_model_output(full_text))
            
        except Exception as e:
            print(f"Error {i}: {e}")
            predictions.append([0] * 75)
    
    predictions = np.array(predictions)
    
    # 4. Calculate metrics
    results_qlora = evaluate_predictions(y_test, predictions, "QLoRA Fine-tuned")
    
    return results_qlora

# 4. Model Loading and Inference Pipeline

In [ ]:
from huggingface_hub import login

login() # logging into Hugging Face to access restricted models like Llama-3.1-8B (not necessary if the model is not used)

In [ ]:
print("Setting up a quantized Llama-3.1-8B-Instruct model with 4 bits...") 

# The larger model is quantized to fit into the T4 GPU's memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Define right padding (standard for training/fine-tuning)
tokenizer.padding_side = "right"

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("Model loaded successfully!\n")

In [ ]:
print("Setting up a quantized Bio-Medical-Llama-3-8B model with 4 bits...")

# The larger model is quantized to fit into the T4 GPU's memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    "ContactDoctor/Bio-Medical-Llama-3-8B",
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("ContactDoctor/Bio-Medical-Llama-3-8B")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("Model loaded successfully!\n")

In [ ]:
print("Setting up the Qwen2.5-3B-Instruct model in FP16...")

# Because this model is smaller than the others, it does not need to be quantized
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16, # Uses medium precision (ideal for T4)
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("Model loaded successfully!\n")

In [ ]:
print("Setting up a quantized Echelon-AI/Med-Qwen2-7B model with 4 bits...")

# The larger model is quantized to fit into the T4 GPU's memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    "Echelon-AI/Med-Qwen2-7B",
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("Echelon-AI/Med-Qwen2-7B")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("Model loaded successfully!\n")

# 5. Final Evaluation: Classical Baseline vs. Base Models vs. QLoRA Trained Models

In [ ]:
model_name = "qwen-3b" # CHOOSE THE NAME OF THE MODEL ['bio-llama', 'llama', 'qwen-3b', 'med-qwen']

In [ ]:
# ============================================================================
# MAIN PIPELINE
# ============================================================================

# 1. Load data
reports, y_labels = load_data("/kaggle/input/mimic-cxr-2/sampled_1000_data.csv")

# 2. Split: 80% train, 20% test (same split as baseline)
reports_train, reports_test, y_train, y_test = train_test_split(
    reports, y_labels, test_size=0.2, random_state=42
)

print(f"Data split:")
print(f"   Train: {len(reports_train)} samples")
print(f"   Test:  {len(reports_test)} samples\n")

# ========================================================================
# SUB-TASK 3(a) - PART 1: 0-SHOT
# ========================================================================

print("\n" + "="*60)
print("STARTING 0-SHOT PROMPTING")
print("="*60)

# Create 0-shot prompts for test set
zero_shot_prompts = [create_zero_shot_prompt(report) for report in reports_test]

# Inference
y_pred_zero = run_inference_with_checkpoint(reports_test, zero_shot_prompts, "0-Shot", model_name)

# Evaluation
results_zero = evaluate_predictions(y_test, y_pred_zero, "0-Shot")
print(y_pred_zero)

# ========================================================================
# SUB-TASK 3(a) - PART 2: 5-SHOT
# ========================================================================

print("\n" + "="*60)
print("STARTING 5-SHOT PROMPTING")
print("="*60)

# Select 5 examples from the train set to use as few-shot examples
np.random.seed(42)
example_indices = np.random.choice(len(reports_train), 5, replace=False)
examples = [(reports_train[i], y_train[i]) for i in example_indices]

print(f"📝 Selected examples for 5-shot: {example_indices}\n")

# Create 5-shot prompts for test set
five_shot_prompts = [create_five_shot_prompt(report, examples) for report in reports_test]

# Inference
y_pred_five = run_inference_with_checkpoint(reports_test, five_shot_prompts, "5-Shot", model_name)

# Evaluation
results_five = evaluate_predictions(y_test, y_pred_five, "5-Shot")

# ========================================================================
# SUB-TASK 3(b) - QLORA FINE-TUNING
# ========================================================================

# ------> RUN THE FOLLOWING LINE ONLY IF THE MODEL STILL HAS NOT BEEN FINETUNED
results_qlora = train_qlora()

# ------> RUN THE FOLLOWING LINE IF THE MODEL HAS ALREADY BEEN FINETUNED
# results_qlora = evaluate_qlora_trained()

# ========================================================================
# FINAL RESULTS WITH QLORA TRAINED MODEL
# ========================================================================

print("\n" + "="*60)
print("📋 FINAL TABLE - SUB-TASKS 3(a) + 3(c)")
print("="*60)

results_df = pd.DataFrame([
    {"Method": "Baseline (TF-IDF + LinearSVC)", "Macro-F1": macro_f1_svc, "Micro-F1": micro_f1_svc},
    {"Method": "Zero-Shot", "Macro-F1": results_zero["macro_f1"], "Micro-F1": results_zero["micro_f1"]},
    {"Method": "5-Shot", "Macro-F1": results_five["macro_f1"], "Micro-F1": results_five["micro_f1"]},
    {"Method": "QLoRA Fine-tuned", "Macro-F1": results_qlora["macro_f1"], "Micro-F1": results_qlora["micro_f1"]}
])

print(results_df.to_string(index=False))
print("="*60 + "\n")

results_df.to_csv("/kaggle/working/subtask3_final_results.csv", index=False)

# UTILITY FUNCTIONS

In [ ]:
# Create download link to fine-tuned model
import os
from IPython.display import FileLink

# 1. Zip the final model folder (qlora_model)
# The command below creates a file called 'final_model.zip'
print("Compressing files...")
!zip -r final_model.zip /kaggle/working/qlora_model

# 2. Generate a direct download link on the notebook.
print("Done! Click the link below to download.:")
FileLink(r'final_model.zip')

In [ ]:
# Function to clean GPU's memory without needing to restart the session
import gc

# 1. Delete large objects
try:
    if 'model' in locals():
        del model
    if 'pipe' in locals():
        del pipe
    if 'tokenizer' in locals():
        del tokenizer
    if 'trainer' in locals():
        del tokenizer
    if 'model_train' in locals():
        del tokenizer
    if 'model_trained' in locals():
        del tokenizer
    if 'results_qlora' in locals():
        del tokenizer
except NameError:
    # Ignore if the variable is not defined
    pass

# 2. Force Python to perform garbage collection.
gc.collect()

# 3. Clear the GPU memory cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory successfully freed.")
else:
    print("CUDA is unavailable. Nothing to clean.")